# Unit 3 Hands-On: Deep Q-Learning (DQN) — SpaceInvaders 실습

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 3**의 실습입니다.  
**DQN(Deep Q-Network)** 알고리즘으로 `SpaceInvadersNoFrameskip-v4` 환경에서  
에이전트를 훈련하고, Hugging Face Hub에 업로드합니다.

### Unit 2와의 차이점

| 항목 | Unit 2 (Q-Learning) | Unit 3 (DQN) |
|---|---|---|
| **Q-함수 표현** | Q-테이블 (표) | 신경망 (CNN) |
| **입력** | 이산 상태 번호 | 게임 화면 픽셀 |
| **환경** | FrozenLake, Taxi | SpaceInvaders (Atari) |
| **구현 도구** | 직접 구현 | RL-Zoo3 (래퍼 라이브러리) |

### RL-Zoo3 란?
Stable-Baselines3 기반의 훈련/평가/업로드 파이프라인을 CLI 명령어로 제공하는 도구입니다.  
복잡한 DQN 구현 없이 `python -m rl_zoo3.train` 한 줄로 훈련을 시작할 수 있습니다.

---
## 목차
1. 환경 설치
2. Google Drive 마운트
3. 가상 디스플레이 설정
4. 훈련 (DQN + SpaceInvaders)
5. 훈련 결과 평가
6. Hugging Face Hub 업로드
7. Hub에서 모델 불러오기 (보너스)


---
## 1. 환경 설치

| 패키지 | 역할 |
|---|---|
| `rl-baselines3-zoo` | DQN 등 알고리즘 훈련/평가/업로드 CLI 제공 |
| `gymnasium[atari]` | Atari 게임 환경 (SpaceInvaders 등) |
| `gymnasium[accept-rom-license]` | Atari ROM 라이선스 자동 수락 |
| `swig`, `cmake`, `ffmpeg` | 빌드 및 영상 처리 의존성 |
| `pyvirtualdisplay` | Colab 가상 디스플레이 |

> ⚠️ 설치 후 런타임 재시작이 필요할 수 있습니다.

In [1]:
# RL-Baselines3-Zoo 설치 (최신 버전 직접 설치)
!pip install git+https://github.com/DLR-RM/rl-baselines3-zoo

  Cloning https://github.com/DLR-RM/rl-baselines3-zoo to /tmp/pip-req-build-mo7q_afw
  Running command git clone --filter=blob:none --quiet https://github.com/DLR-RM/rl-baselines3-zoo /tmp/pip-req-build-mo7q_afw
  Resolved https://github.com/DLR-RM/rl-baselines3-zoo to commit f94cef4e59fb2f03b77177db432c3f771d0ee72c
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 21.1 MB/s eta 0:00:00
  Cr

In [2]:
# 빌드 및 영상 처리 시스템 패키지 설치
!apt-get install -y swig cmake ffmpeg -qq

Selecting previously unselected package swig4.0.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubuntu1_all.deb ...
Unpacking swig (4.0.2-1ubuntu1) ...
Setting up swig4.0 (4.0.2-1ubuntu1) ...
Setting up swig (4.0.2-1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
# Atari 게임 환경 및 ROM 라이선스 수락
!pip install 'gymnasium[atari]' -q
!pip install 'gymnasium[accept-rom-license]' -q

In [4]:
# 가상 디스플레이 패키지 설치 (Colab은 물리 모니터가 없으므로 필요)
!apt install -y python-opengl ffmpeg xvfb -qq
!pip3 install pyvirtualdisplay -q

E: Unable to locate package python-opengl


---
## 2. Google Drive 마운트

Colab VM은 세션 종료 시 파일이 모두 삭제됩니다.  
훈련 로그와 모델을 Google Drive에 저장하여 영구 보존합니다.

```
Google Drive/RL_Course/Unit3_DQN/
└── logs/
    └── dqn/
        └── SpaceInvadersNoFrameskip-v4_1/
            ├── best_model.zip      ← 훈련 중 최고 성능 모델
            ├── evaluations.npz     ← 평가 기록
            └── SpaceInvadersNoFrameskip-v4/
                └── replay.mp4      ← 최종 플레이 영상
```

> 실행 시 Google 계정 인증 팝업이 뜹니다. 허용해주세요.

In [5]:
from google.colab import drive
import os

# Google Drive 마운트
drive.mount('/content/drive')

# ✏️ 저장 폴더명을 원하는 대로 변경하세요.
DRIVE_BASE = '/content/drive/MyDrive/RL_Course/Unit3_DQN'
LOG_DIR    = f'{DRIVE_BASE}/logs'

os.makedirs(LOG_DIR, exist_ok=True)

print('✅ Drive 마운트 완료!')
print(f'   로그/모델 저장 경로 : {LOG_DIR}')


Mounted at /content/drive
✅ Drive 마운트 완료!
   로그/모델 저장 경로 : /content/drive/MyDrive/RL_Course/Unit3_DQN/logs


---
## 3. 가상 디스플레이 설정

Colab에는 물리적인 화면이 없으므로 가상 디스플레이를 생성합니다.  
RL-Zoo3의 `--no-render` 옵션으로 훈련 중 렌더링은 생략하고,  
평가 시에만 영상 파일로 저장합니다.

In [6]:
from pyvirtualdisplay import Display

# 보이지 않는 가상 디스플레이 시작 (1400×900)
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print('✅ 가상 디스플레이 시작')


✅ 가상 디스플레이 시작


---
## 4. 훈련 (DQN + SpaceInvaders)

### DQN 이란?

**Deep Q-Network**: Q-테이블 대신 **CNN(합성곱 신경망)**으로 Q값을 근사합니다.  
게임 화면(픽셀)을 입력으로 받아 각 행동의 Q값을 출력합니다.

```
게임 화면 (픽셀)
     │
     ▼
  CNN 레이어들   ← 화면에서 특징 추출
     │
     ▼
  FC 레이어들
     │
     ▼
  Q값 출력 [Q(s,좌), Q(s,우), Q(s,발사), ...]
```

### DQN 핵심 기법

| 기법 | 설명 |
|---|---|
| **Experience Replay** | 과거 경험을 버퍼에 저장 후 랜덤 샘플링 → 데이터 상관성 감소 |
| **Target Network** | 별도의 타겟 네트워크로 안정적인 학습 목표 유지 |
| **Frame Stacking** | 연속 4프레임을 쌓아 속도/방향 정보 제공 |

### 4-1. dqn.yml 파일 생성

`-c dqn.yml` 옵션으로 전달하는 하이퍼파라미터 설정 파일입니다.  
RL-Zoo3는 이 파일을 읽어 훈련 설정을 적용합니다.

| 파라미터 | 값 | 의미 |
|---|---|---|
| `n_timesteps` | 1e7 (1,000만) | 총 훈련 스텝 수 |
| `buffer_size` | 100,000 | 경험 리플레이 버퍼 크기 |
| `learning_rate` | 1e-4 | 학습률 |
| `batch_size` | 32 | 미니배치 크기 |
| `learning_starts` | 100,000 | 몇 스텝 후 학습 시작 (버퍼 채우기) |
| `target_update_interval` | 1,000 | 타겟 네트워크 업데이트 주기 |
| `train_freq` | 4 | 4스텝마다 한 번 학습 |
| `exploration_fraction` | 0.1 | 전체의 10% 동안 ε 감소 |
| `exploration_final_eps` | 0.01 | 최종 탐색률 1% |
| `frame_stack` | 4 | 연속 4프레임 스택 |
| `policy` | CnnPolicy | CNN 기반 정책 네트워크 |

> ⏱️ 예상 훈련 시간: Colab T4 GPU 기준 **약 60~90분**  
> 훈련 중 로그에서 `ep_rew_mean`이 올라가는지 확인하세요.

In [ ]:
# dqn.yml 파일 생성
# RL-Zoo3가 이 파일을 읽어 훈련 하이퍼파라미터를 적용합니다.
# ✏️ 하이퍼파라미터를 변경하고 싶다면 아래 값을 수정하세요.
dqn_yml = """\
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4                  # 연속 4프레임을 쌓아 속도/방향 정보 제공
  policy: 'CnnPolicy'             # CNN 기반 정책 네트워크 (픽셀 입력)
  n_timesteps: !!float 5e5        # 총 훈련 스텝: 500,000
  buffer_size: 100000             # 경험 리플레이 버퍼 크기
  learning_rate: !!float 1e-4     # 학습률
  batch_size: 32                  # 미니배치 크기
  learning_starts: 100000         # 100,000 스텝 동안 버퍼 채운 후 학습 시작
  target_update_interval: 1000    # 1,000 스텝마다 타겟 네트워크 업데이트
  train_freq: 4                   # 4 스텝마다 1회 학습
  gradient_steps: 1               # 업데이트당 그래디언트 스텝 수
  exploration_fraction: 0.1       # 전체 스텝의 10% 동안 ε 감소
  exploration_final_eps: 0.01     # 최종 탐색률 1%
  optimize_memory_usage: False    # 메모리 최적화 (True 시 replay buffer 설정 변경 필요)
"""

# 파일 저장
with open('dqn.yml', 'w') as f:
    f.write(dqn_yml)

print('✅ dqn.yml 생성 완료')
print('\n파일 내용:')
print(dqn_yml)


✅ dqn.yml 생성 완료

파일 내용:
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4                  # 연속 4프레임을 쌓아 속도/방향 정보 제공
  policy: 'CnnPolicy'             # CNN 기반 정책 네트워크 (픽셀 입력)
  n_timesteps: !!float 5e5        # 총 훈련 스텝: 500,000
  buffer_size: 100000             # 경험 리플레이 버퍼 크기
  learning_rate: !!float 1e-4     # 학습률
  batch_size: 32                  # 미니배치 크기
  learning_starts: 100000         # 100,000 스텝 동안 버퍼 채운 후 학습 시작
  target_update_interval: 1000    # 1,000 스텝마다 타겟 네트워크 업데이트
  train_freq: 4                   # 4 스텝마다 1회 학습
  gradient_steps: 1               # 업데이트당 그래디언트 스텝 수
  exploration_fraction: 0.1       # 전체 스텝의 10% 동안 ε 감소
  exploration_final_eps: 0.01     # 최종 탐색률 1%
  optimize_memory_usage: False    # 메모리 최적화 (True 시 replay buffer 설정 변경 필요)



### 4-2. 훈련 실행

```bash
python -m rl_zoo3.train \
    --algo dqn                          # 알고리즘: DQN
    --env SpaceInvadersNoFrameskip-v4   # 환경: SpaceInvaders
    -f {LOG_DIR}                        # 로그/모델 저장 폴더 (Drive)
    -c dqn.yml                          # 하이퍼파라미터 설정 파일
```


In [11]:
# DQN으로 SpaceInvaders 훈련
# -f 옵션에 Drive 경로를 지정하여 결과를 Drive에 직접 저장
# -c dqn.yml: 위에서 생성한 하이퍼파라미터 파일 사용
!python -m rl_zoo3.train \
    --algo dqn \
    --env SpaceInvadersNoFrameskip-v4 \
    -f {LOG_DIR} \
    -c dqn.yml \
    --save-freq 100000


2026-08-02 16:51:32.548679: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
========== SpaceInvadersNoFrameskip-v4 ==========
Seed: 1949699820
Loading hyperparameters from: dqn.yml
Default hyperparameters for environment (ones being tuned will be overridden):
OrderedDict([('batch_size', 32),
             ('buffer_size', 100000),
             ('env_wrapper',
              ['stable_baselines3

---
## 5. 훈련 결과 평가

훈련이 완료된 모델을 `rl_zoo3.enjoy`로 평가합니다.  
`--no-render`: 창을 띄우지 않고 백그라운드에서 실행 (Colab 환경)  
`--n-timesteps 5000`: 5,000 스텝 동안 에이전트 실행

평가 결과로 평균 점수가 출력됩니다. SpaceInvaders 기준:

| 평균 점수 | 수준 |
|---|---|
| 300 미만 | 초반 학습 수준 |
| 300 ~ 600 | 기본적인 플레이 가능 |
| **600 이상** | **잘 학습된 에이전트** |

In [12]:
# 훈련된 모델 평가 (5,000 스텝)
# --no-render: 창 없이 백그라운드 실행 → 커널 크래시 방지
!python -m rl_zoo3.enjoy \
    --algo dqn \
    --env SpaceInvadersNoFrameskip-v4 \
    --no-render \
    --n-timesteps 5000 \
    --folder {LOG_DIR}


2026-08-02 17:22:59.391741: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Loading latest experiment, id=3
Loading /content/drive/MyDrive/RL_Course/Unit3_DQN/logs/dqn/SpaceInvadersNoFrameskip-v4_3/SpaceInvadersNoFrameskip-v4.zip
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Stacking 4 frames
Atari Episode Score: 285.00
Atari Episode Length 2345
Atari Episo

---
## 5-1. 체크포인트별 영상 저장

RL-Zoo3는 훈련 중 `--save-freq` 옵션으로 주기적으로 체크포인트를 저장합니다.  
훈련 완료 후 저장된 체크포인트들을 순서대로 불러와 각각 mp4로 저장합니다.

```
logs/dqn/SpaceInvadersNoFrameskip-v4_1/
├── best_model.zip          ← 훈련 중 최고 성능 모델
├── rl_model_100000_steps.zip
├── rl_model_200000_steps.zip   ← 체크포인트들
├── rl_model_300000_steps.zip
└── ...
```

### 5-1-2. 체크포인트별 영상 생성

저장된 체크포인트를 순서대로 불러와 각각 1 에피소드를 실행하고 mp4로 저장합니다.  
영상 파일명에 스텝 수가 포함되어 학습 과정을 시간 순서대로 비교할 수 있습니다.

```
Google Drive/RL_Course/Unit3_DQN/training_videos/
├── step_0100000.mp4
├── step_0200000.mp4
├── ...
├── step_1000000.mp4
└── best_model.mp4   ← 훈련 중 최고 성능
```


In [23]:
import glob, os, re
import numpy as np
import imageio
import gymnasium as gym
import ale_py

# ── 핵심 해결 코드: 아타리 환경 명시적 등록 ──────────────────────
gym.register_envs(ale_py)
# ─────────────────────────────────────────────────────────────────

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from IPython.display import Video, display

# ── 경로 설정 ────────────────────────────────────────────────────
ENV_ID   = 'SpaceInvadersNoFrameskip-v4'
CKPT_DIR = f'{LOG_DIR}/dqn/{ENV_ID}_3'
VID_DIR  = f'{DRIVE_BASE}/training_videos'
os.makedirs(VID_DIR, exist_ok=True)


def make_env():
    """평가용 Atari 환경 생성 (Frame Stack 4 적용)"""
    env = make_atari_env(ENV_ID, n_envs=1, seed=42)
    env = VecFrameStack(env, n_stack=4)
    return env


def record_model(model_path, video_path, n_steps=1000, fps=30):
    """주어진 모델로 에피소드를 실행하고 mp4로 저장"""
    env = make_env()
    model = DQN.load(model_path, env=env)

    frames = []
    obs = env.reset()
    # VecEnv는 render()가 없으므로 get_images() 사용
    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, dones, _ = env.step(action)
        frame = env.render(mode='rgb_array')
        frames.append(frame)
        if dones[0]:
            break

    env.close()
    imageio.mimsave(video_path, frames, fps=fps)


# ── 체크포인트 목록 수집 및 정렬 ────────────────────────────────
ckpt_files = sorted(
    glob.glob(f'{CKPT_DIR}/rl_model_*_steps.zip'),
    key=lambda x: int(re.search(r'rl_model_(\d+)_steps', x).group(1))
)
best_model = f'{CKPT_DIR}/best_model.zip'

print(f'체크포인트 {len(ckpt_files)}개 발견')
for f in ckpt_files:
    print(' ', os.path.basename(f))

# ── 체크포인트별 영상 생성 ────────────────────────────────────────
for ckpt_path in ckpt_files:
    steps = int(re.search(r'rl_model_(\d+)_steps', ckpt_path).group(1))
    video_path = f'{VID_DIR}/step_{steps:08d}.mp4'

    if os.path.exists(video_path):
        print(f'  ⏭️  [{steps:,} 스텝] 이미 존재, 건너뜀')
        continue

    print(f'  🎬 [{steps:,} 스텝] 영상 생성 중...')
    try:
        record_model(ckpt_path, video_path)
        print(f'  ✅ 저장 완료: {video_path}')
    except Exception as e:
        print(f'  ⚠️  실패: {e}')

# ── best_model 영상 생성 ─────────────────────────────────────────
if os.path.exists(best_model):
    best_video = f'{VID_DIR}/best_model.mp4'
    print('\n🏆 best_model 영상 생성 중...')
    record_model(best_model, best_video)
    print(f'✅ 저장 완료: {best_video}')

print('\n✅ 모든 영상 생성 완료!')
print(f'   저장 위치: {VID_DIR}')

체크포인트 5개 발견
  rl_model_100000_steps.zip
  rl_model_200000_steps.zip
  rl_model_300000_steps.zip
  rl_model_400000_steps.zip
  rl_model_500000_steps.zip
  🎬 [100,000 스텝] 영상 생성 중...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Wrapping the env in a VecTransposeImage.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  ✅ 저장 완료: /content/drive/MyDrive/RL_Course/Unit3_DQN/training_videos/step_00100000.mp4
  🎬 [200,000 스텝] 영상 생성 중...
Wrapping the env in a VecTransposeImage.


  ✅ 저장 완료: /content/drive/MyDrive/RL_Course/Unit3_DQN/training_videos/step_00200000.mp4
  🎬 [300,000 스텝] 영상 생성 중...
Wrapping the env in a VecTransposeImage.


  ✅ 저장 완료: /content/drive/MyDrive/RL_Course/Unit3_DQN/training_videos/step_00300000.mp4
  🎬 [400,000 스텝] 영상 생성 중...
Wrapping the env in a VecTransposeImage.


  ✅ 저장 완료: /content/drive/MyDrive/RL_Course/Unit3_DQN/training_videos/step_00400000.mp4
  🎬 [500,000 스텝] 영상 생성 중...
Wrapping the env in a VecTransposeImage.


  ✅ 저장 완료: /content/drive/MyDrive/RL_Course/Unit3_DQN/training_videos/step_00500000.mp4

🏆 best_model 영상 생성 중...
Wrapping the env in a VecTransposeImage.


✅ 저장 완료: /content/drive/MyDrive/RL_Course/Unit3_DQN/training_videos/best_model.mp4

✅ 모든 영상 생성 완료!
   저장 위치: /content/drive/MyDrive/RL_Course/Unit3_DQN/training_videos


### 5-1-3. 영상 순서대로 재생

저장된 영상을 시간 순서대로 재생하여 학습 과정을 확인합니다.


In [24]:
# 저장된 영상 목록 출력
videos = sorted(glob.glob(f'{VID_DIR}/step_*.mp4'))
best_vid = f'{VID_DIR}/best_model.mp4'

print(f'총 {len(videos)}개의 체크포인트 영상 + best_model')

# 체크포인트 영상 순서대로 재생
for video_path in videos:
    steps = int(re.search(r'step_(\d+)', video_path).group(1))
    print(f'\n📽️  {steps:,} 스텝 시점')
    display(Video(video_path, embed=True, width=400))

# best_model 영상 재생
if os.path.exists(best_vid):
    print('\n🏆 Best Model')
    display(Video(best_vid, embed=True, width=400))


총 5개의 체크포인트 영상 + best_model

📽️  100,000 스텝 시점



📽️  200,000 스텝 시점



📽️  300,000 스텝 시점



📽️  400,000 스텝 시점



📽️  500,000 스텝 시점



🏆 Best Model


## 참고 사항 : 학습 타임스텝(Time Step) 조절

이번 실습은 빠른 진행을 위해 **50만 번(5e5)**만 학습하도록 설정되었습니다. 하지만 **SpaceInvaders**와 같은 복잡한 게임 환경에서 에이전트가 제대로 된 전략을 학습하려면 **최소 100만 번 이상**의 훈련이 필요합니다. 

시간적 여유가 있으실 때 타임스텝을 늘려서 다시 시도해 보시는 것을 권장합니다.

> **📌 설정 변경 방법**  
> `dqn.yml` 파일에서 `n_timesteps` 값을 아래와 같이 조정하시면 됩니다.  
> (예: 100만 번 학습 시 `!!float 1e6`으로, 500만 번 학습 시 `!!float 5e6`으로 변경)
>
> ```yaml
> n_timesteps: !!float 1e6
> ```

---
## 6. Hugging Face Hub 업로드

훈련된 모델을 HF Hub에 업로드합니다.  
RL-Zoo3의 `push_to_hub` 명령어가 모델 카드, 평가 결과, 영상까지 자동으로 생성합니다.

### 사전 준비
1. [Hugging Face 계정 생성](https://huggingface.co/join)
2. [쓰기(write) 권한 토큰 발급](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰으로 교체하세요
# ⚠️ 토큰은 절대 외부에 공개하지 마세요!
login(token='hf_xxxxxxxxxxxxxxxxxxxxxxxx')

# git credential 저장 (push_to_hub 명령어에 필요)
!git config --global credential.helper store


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
# ✏️ 아래 두 값을 본인 정보로 수정하세요.
USERNAME  = 'DitDahDitDit'   # ← HF 사용자명
REPO_NAME = 'dqn-SpaceInvadersNoFrameskip-v4'

# RL-Zoo3가 평가 → 영상 생성 → Hub 업로드를 자동으로 처리
!python -m rl_zoo3.push_to_hub \
    --algo dqn \
    --env SpaceInvadersNoFrameskip-v4 \
    --repo-name {REPO_NAME} \
    -orga {USERNAME} \
    -f {LOG_DIR}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Loading latest experiment, id=3
Loading /content/drive/MyDrive/RL_Course/Unit3_DQN/logs/dqn/SpaceInvadersNoFrameskip-v4_3/SpaceInvadersNoFrameskip-v4.zip
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Stacking 4 frames
Wrapping the env in a VecTransposeImage.
Uploading to DitDahDitDit/dqn-SpaceInvadersNoFrameskip-v4, make sure to have the rights
ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to some
minutes if video generation is activated. This is a work in progress: if you
encounter a bug, please open an i